# Notebook 09 — Peer Review

**Module:** 18 — Scientific Writing and Open Science  
**Tier:** 2 — Working competence  
**Notebook:** 9 of 12  
**Time estimate:** 75 minutes

> Peer review is the quality filter for published science. Understanding
> how to write a review, and how to respond to one, are skills that
> separate junior from senior researchers. Both sides of the process
> are covered here.

---
## Step 1 — Motivation

You will encounter peer review from three angles:
1. **As an author:** receiving reviews and writing revision letters
2. **As a reviewer:** evaluating others' work (increasingly common from PhD Year 2)
3. **As a reader:** deciding how much weight to give a peer-reviewed vs. preprint result

Supervisors frequently ask PhD students to assist with or write first drafts of reviews.
Writing a good review is a professional signal of your domain competence.

---
## Step 2 — Writing a Review: Structure

A review has four parts:

| Part | Length | Purpose |
|------|--------|--------|
| **Summary** | 3–5 sentences | Prove you read the paper by summarizing it accurately |
| **Major concerns** | Numbered list | The 2–4 issues that must be addressed before publication |
| **Minor concerns** | Numbered list | Small issues: typos, unclear figures, missing references |
| **Recommendation** | 1 sentence | Accept / Minor revision / Major revision / Reject |

**Review tone:** critique the science, not the authors. "The description of the
cross-validation strategy on page 4 is unclear — it is not evident whether the
split is performed before or after feature selection" is a valid technical concern.
"The authors do not understand cross-validation" is not.

**Major vs. minor concerns:**
- **Major:** issues that could change the conclusion (missing baseline, statistical
  error, wrong control, missing dataset, overclaiming).
- **Minor:** issues that don't change the conclusion (figure labels, grammar,
  missing citation, unclear terminology).

---
## Step 3 — Common Reviewer Mistakes

1. **Asking for experiments beyond the paper's scope.** If the paper is a method
   paper, asking for clinical validation is out of scope unless the paper itself
   claims clinical relevance.
2. **Vague concerns without actionable requests.** "The analysis is insufficient"
   is not actionable. "The authors should report the 95% confidence interval for
   the mean AUROC, and test whether the improvement is significant by a paired
   Wilcoxon test (n = number of TFs)" is.
3. **Asking for the authors' personal data.** You review the paper, not the lab's
   reputation.
4. **Taking months to review.** The standard is 3–4 weeks. Delays harm authors.

---
## Step 4 — Responding to Reviewers: The Revision Letter

Structure of a revision letter:

1. **Opening:** "We thank the reviewers for their thorough evaluation. We have
   addressed all concerns as detailed below."
2. **For each reviewer concern:** copy the concern in italic, then respond.
   - If you agree and fixed it: state what you changed and where in the manuscript.
   - If you partially agree: explain what you did and why you didn't do more.
   - If you disagree: provide a technical rebuttal with evidence or citations.
3. **Closing:** summary of what changed.

**Golden rule:** every reviewer concern gets a response. "We thank the reviewer
for this suggestion" followed by no action is the most common cause of rejection
at the revision stage.

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ---- Review quality analysis tools ----

def audit_review(review_text):
    """
    Heuristically score a peer review for quality.
    """
    t = review_text.lower()
    sentences = [s.strip() for s in re.split(r'[.!?]', t) if len(s.strip()) > 10]

    # Summary present?
    summary_present = any(kw in t for kw in ['the paper', 'the authors', 'this study',
                                                 'the manuscript', 'the work'])
    # Numbered concerns?
    has_numbered = bool(re.search(r'^\d+[\.\)]', review_text, re.MULTILINE))
    # Actionable requests?
    actionable_words = ['should', 'must', 'please', 'add', 'report', 'include',
                          'clarify', 'provide', 'compare', 'test', 'show']
    actionable_count = sum(1 for w in actionable_words if w in t)
    # Personal attacks?
    attack_words = ['incompetent', 'do not understand', "don't understand",
                      'careless', 'sloppy', 'naive', 'trivial']
    attack_found = [w for w in attack_words if w in t]
    # Out-of-scope requests?
    scope_words = ['clinical trial', 'animal study', 'in vivo', 'patients',
                     'not the focus', 'beyond the scope']
    out_of_scope = sum(1 for w in scope_words if w in t)
    # Vague concerns ("insufficient", "poor quality" without specifics)?
    vague_words = ['insufficient', 'not enough', 'poor quality', 'lacks rigor',
                     'is missing', 'could be better']
    vague_count = sum(1 for w in vague_words if w in t)

    return {
        'summary_present': summary_present,
        'has_numbered_list': has_numbered,
        'actionable_count': actionable_count,
        'personal_attacks': attack_found,
        'vague_concern_count': vague_count,
        'out_of_scope_refs': out_of_scope,
        'word_count': len(review_text.split()),
    }

REVIEW_WEAK = """
This paper attempts to address TF binding prediction. I have read the paper.
Major concerns:
1. The paper is insufficient and lacks rigor.
2. The authors do not understand cross-validation.
3. The results could be better.
4. A clinical trial should be performed to validate these results.
Minor:
- Typos on page 3.
This paper should be rejected.
"""

REVIEW_STRONG = """
This study presents DeepBind-v2, a convolutional neural network for TF binding prediction
trained on 690 ENCODE ChIP-seq datasets. The authors achieve AUROC 0.924 on the ENCODE
benchmark, outperforming the PWM baseline by 12.3 pp. The method is clearly described
and the code is made available. The work represents a meaningful advance over position
weight matrix approaches.

Major concerns:
1. Cross-validation strategy is unclear. The authors state they use an 80/10/10
   train/validation/test split but do not specify whether the split is performed
   before or after data augmentation. If augmentation precedes splitting, the
   test set may be contaminated. The authors should clarify this and, if necessary,
   re-evaluate on a properly held-out test set.
2. Multiple testing correction. 690 TFs are compared, but no correction for multiple
   comparisons is described. The authors should apply Bonferroni or Benjamini-Hochberg
   correction and report adjusted p-values.

Minor concerns:
1. Figure 2 caption does not state how error bars were computed (SD or SEM).
2. Line 147: "data is" should be "data are".
3. The paper should cite Kelley et al. (2016) which describes a similar approach.

Overall, I recommend major revision. The two major concerns are straightforward to
address and do not require new experiments.
"""

for name, rev in [('WEAK', REVIEW_WEAK), ('STRONG', REVIEW_STRONG)]:
    result = audit_review(rev)
    print(f'\n=== {name} review ===')
    for k, v in result.items():
        print(f'  {k}: {v}')

In [ ]:
# Revision letter template generator
def generate_revision_letter(paper_title, reviewer_concerns):
    """
    Generate a skeleton revision letter.
    reviewer_concerns: list of (reviewer_id, concern_text, response_text) tuples
    """
    lines = [
        f'# Response to reviewers: {paper_title}',
        '',
        'We thank the reviewers for their thorough evaluation of our manuscript. '
        'We have carefully considered all comments and addressed each point below. '
        'Changes to the manuscript are indicated in bold in the revised version.',
        '',
    ]
    current_reviewer = None
    for r_id, concern, response in reviewer_concerns:
        if r_id != current_reviewer:
            lines.append(f'## Reviewer {r_id}')
            lines.append('')
            current_reviewer = r_id
        lines.append(f'**Reviewer concern:** *{concern}*')
        lines.append('')
        lines.append(f'**Response:** {response}')
        lines.append('')
    return '\n'.join(lines)

# Example
letter = generate_revision_letter(
    'DeepBind-v2: CNN-based TF binding prediction',
    [
        (1, 'Cross-validation strategy is unclear.',
         'We thank the reviewer for identifying this ambiguity. The 80/10/10 split '
         'was performed before augmentation; no test set contamination occurred. '
         'We have clarified this in the revised Methods section (lines 145–152) and '
         'added a data flow diagram as Supplementary Figure 1.'),
        (1, 'Multiple testing correction is missing.',
         'We have applied Bonferroni correction for all 690 TF comparisons. '
         'The improvement remains significant at α = 0.05 (adjusted p < 0.001 '
         'for 98.4% of TFs). Table 1 has been updated with adjusted p-values.'),
        (2, 'Figure 2 caption does not state error bar type.',
         'We have added "error bars = 1 SD" to the Figure 2 caption (line 238). '
         'We also added the number of independent replicates (n = 5) to the caption.'),
    ]
)
print(letter[:1200] + '\n...(truncated)' if len(letter) > 1200 else letter)

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 7))

# Panel A: Review quality audit
ax = axes[0]
criteria_rev = ['Summary\npresent', 'Numbered\nlist', 'Actionable\n(≥3)', 'No personal\nattacks', 'No vague\nconcerns']

weak_r   = audit_review(REVIEW_WEAK)
strong_r = audit_review(REVIEW_STRONG)

weak_vals   = [
    int(weak_r['summary_present']),
    int(weak_r['has_numbered_list']),
    min(1, weak_r['actionable_count'] / 3),
    1 - min(1, len(weak_r['personal_attacks'])),
    max(0, 1 - weak_r['vague_concern_count'] / 2),
]
strong_vals = [
    int(strong_r['summary_present']),
    int(strong_r['has_numbered_list']),
    min(1, strong_r['actionable_count'] / 3),
    1 - min(1, len(strong_r['personal_attacks'])),
    max(0, 1 - strong_r['vague_concern_count'] / 2),
]

x = np.arange(len(criteria_rev))
w = 0.35
ax.bar(x - w/2, weak_vals,   w, label='Weak review',   color='tomato',    edgecolor='black', alpha=0.8)
ax.bar(x + w/2, strong_vals, w, label='Strong review', color='steelblue', edgecolor='black', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(criteria_rev, fontsize=8)
ax.set_ylim(0, 1.2); ax.set_ylabel('Score (1.0 = meets criterion)')
ax.set_title('A. Review quality audit')
ax.legend(fontsize=9)
ax.axhline(1.0, color='grey', ls=':', lw=0.8)

# Panel B: Review response decision tree
ax = axes[1]
ax.axis('off')
nodes = [
    (0.5, 0.92, 'Reviewer concern received', '#4e79a7', 0.18),
    (0.2, 0.70, 'Do you agree?', '#59a14f', 0.14),
    (0.8, 0.70, 'Do you disagree?', '#e15759', 0.14),
    (0.2, 0.48, 'Fix it + say where', '#59a14f', 0.14),
    (0.5, 0.48, 'Partial: fix what\nyou can, explain\nthe rest', '#f28e2b', 0.14),
    (0.8, 0.48, 'Rebut with\nevidence / citations', '#e15759', 0.14),
    (0.5, 0.22, 'Every concern gets\na response — no exceptions', '#4e79a7', 0.14),
]
for (x, y, text, color, w_box) in nodes:
    ax.add_patch(mpatches.FancyBboxPatch((x-w_box/2, y-0.06), w_box, 0.12,
                                           boxstyle='round,pad=0.02', facecolor=color,
                                           alpha=0.3, transform=ax.transAxes))
    ax.text(x, y, text, transform=ax.transAxes, fontsize=8, ha='center', va='center')
for (x1, y1, x2, y2) in [(0.5, 0.86, 0.2, 0.76), (0.5, 0.86, 0.5, 0.54),
                            (0.5, 0.86, 0.8, 0.76), (0.2, 0.64, 0.2, 0.54),
                            (0.8, 0.64, 0.8, 0.54),
                            (0.2, 0.42, 0.5, 0.28), (0.5, 0.42, 0.5, 0.28),
                            (0.8, 0.42, 0.5, 0.28)]:
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                  xycoords='axes fraction', textcoords='axes fraction',
                  arrowprops=dict(arrowstyle='->', color='grey', lw=1.2))
ax.set_title('B. Revision response decision tree')

# Panel C: Major vs minor concerns classifier
ax = axes[2]
ax.axis('off')
major_examples = [
    'Missing baseline comparison',
    'No multiple testing correction',
    'Data leakage in CV split',
    'Overclaiming in conclusions',
    'Missing negative control',
]
minor_examples = [
    'Figure label missing units',
    'Typo on page 3',
    'Undefined abbreviation',
    'Missing reference to X et al.',
    'Figure caption incomplete',
]
for i, item in enumerate(major_examples):
    y = 0.9 - i * 0.12
    ax.text(0.04, y, '⚠ Major', transform=ax.transAxes, fontsize=8,
              color='tomato', fontweight='bold', va='center')
    ax.text(0.22, y, item, transform=ax.transAxes, fontsize=8, va='center')
for i, item in enumerate(minor_examples):
    y = 0.9 - i * 0.12
    ax.text(0.54, y, '✓ Minor', transform=ax.transAxes, fontsize=8,
              color='steelblue', fontweight='bold', va='center')
    ax.text(0.72, y, item, transform=ax.transAxes, fontsize=8, va='center')
ax.axvline(0.5, ymin=0.08, ymax=0.98, color='grey', ls='--', lw=1, transform=ax.transAxes)
ax.set_title('C. Major vs. minor review concerns\n(examples from computational biology)')

plt.suptitle('Module 18 NB09: Peer Review', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('peer_review.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

See E09 in `exercises/README.md`: write a structured review of the
weak review above — what are its major and minor problems?

---
## Step 10 — Quiz

1. What are the four parts of a structured peer review? What does each contain?
2. What is the difference between a major and a minor reviewer concern?
   Give one example of each for a machine learning methods paper.
3. How do you respond to a reviewer concern you disagree with?
   What should your response contain?
4. A reviewer requests "in vivo validation in mice" for a bioinformatics
   methods paper that makes no in vivo claims. Is this a valid review concern?
   How would you respond as an author?

---
## Step 12 — Reflection

> *[Write a 3-paragraph review of NB05 from any prior module in this curriculum:
> (1) a 2-sentence summary of what it does, (2) one major concern, (3) one minor
> concern. Use the structure: summary → major → minor → recommendation. Then
> run `audit_review()` on your own review.]*

---
*Next: `10_phd_application_writing.ipynb`*